# Lecture: PPO from stable-baselines3

**Objective**: Apply the Proximal Policy Optimization (PPO) implementation of
`stable-baselines3` (SB3) to the CartPole environment we already solved with
REINFORCE, and &mdash; more importantly &mdash; learn to **read and interpret
the training metrics**:

- which metrics matter,
- what value ranges are *normal*,
- what a value tells you about the health of training,
- and which **extensions** exist (network architecture, MLP vs. LSTM policies).

If you have not done so yet, work through `80-KL-Divergence.ipynb` first: the
`approx_kl` metric below is exactly the $D_{KL}(\pi_{old}\,\|\,\pi_{new})$ from
that notebook.

Run the following cell **only on Google Colab** to install
`stable-baselines3`. If you work locally and already installed it (manually or
via `requirements.txt`), skip this cell.

In [ ]:
!pip install stable-baselines3==2.6.0

### Exercise 1: Train PPO and read the training log

The cell below trains SB3's PPO on `CartPole-v1` with the default
**`MlpPolicy`** (a small multilayer perceptron). With `verbose=1`, PPO prints a
table every `n_steps` update. Run it and watch the numbers change.

**Task**: While training runs, look at the printed table and figure out what
each entry means. The full reference is given right after the cell &mdash; but
try to form a hypothesis first.

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# Create the environment
env = gym.make("CartPole-v1")

# Create a PPO agent with an MLP policy. tensorboard_log is optional (see below).
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./ppo_cartpole_tb/")

# Train the model
model.learn(total_timesteps=100_000)

# Evaluate the trained policy over 10 episodes
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"\nMean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

# Save the model
model.save("ppo_cartpole")


### Reference: what the PPO training metrics mean

SB3 groups the printed values into `rollout/`, `time/` and `train/`. On
**CartPole-v1** the maximum return is **500** (the episode length cap), so a
healthy run drives `ep_rew_mean` towards 500.

#### `rollout/` &mdash; behaviour in the environment

| Metric | Meaning | Normal range on CartPole | What to watch for |
|--------|---------|--------------------------|-------------------|
| `ep_rew_mean` | Mean episodic return over recent episodes | starts ~20&ndash;30, should climb toward **500** | Flat/decreasing &rarr; learning stalled or diverged |
| `ep_len_mean` | Mean episode length | equal to `ep_rew_mean` here (reward = +1 per step) | Same signal as reward for CartPole |

#### `time/` &mdash; bookkeeping

| Metric | Meaning | Notes |
|--------|---------|-------|
| `fps` | Environment steps per second | Hardware-dependent; only relative changes matter |
| `iterations` | Number of rollout+update cycles so far | &mdash; |
| `total_timesteps` | Environment steps collected so far | Progress toward your budget |

#### `train/` &mdash; the optimisation itself (the important diagnostics)

| Metric | Meaning | Normal / healthy range | What it tells you |
|--------|---------|------------------------|-------------------|
| `approx_kl` | Est. $D_{KL}(\pi_{old}\,\|\,\pi_{new})$ per update | **~0.003&ndash;0.02** | Too large (&gt;~0.05) &rarr; updates too aggressive, policy jumps; too small &rarr; barely learning |
| `clip_fraction` | Fraction of samples whose ratio was clipped | **~0.05&ndash;0.2** | High &rarr; policy wants to move far, clipping is doing heavy lifting |
| `clip_range` | The PPO clip parameter $\epsilon$ | constant **0.2** by default | Only changes if you schedule it |
| `entropy_loss` | Negative mean policy entropy | starts near **&minus;0.69** (=&minus;ln 2 for 2 actions), rises toward 0 | Drops to 0 too fast &rarr; premature determinism, too little exploration |
| `explained_variance` | How well the value function predicts returns | should rise toward **~1.0** | Near 0 or negative &rarr; critic is useless; &lt;0 means worse than predicting the mean |
| `learning_rate` | Current optimizer step size | constant **3e-4** by default | Changes only if you schedule it |
| `loss` | Total combined loss (policy + value + entropy) | noisy, no absolute target | Trend matters more than the value; not a clean "lower is better" |
| `policy_gradient_loss` | The clipped surrogate policy loss | small, usually **negative** | &mdash; |
| `value_loss` | Critic MSE on returns | env-dependent, should **trend down** | Exploding &rarr; value targets unstable |
| `n_updates` | Total gradient updates so far | &mdash; | &mdash; |
| `std` | (continuous actions only) policy stddev | shrinks as policy sharpens | Not shown for discrete CartPole |

**Rules of thumb for a healthy PPO run:**
- `ep_rew_mean` rises steadily toward the environment maximum.
- `approx_kl` stays small and roughly constant (the trust region is respected).
- `explained_variance` climbs toward 1 (the critic is learning).
- `entropy_loss` rises *gradually* toward 0 (exploration decays smoothly, not abruptly).

### Exercise 2: Log the metrics and plot them (works locally and on Colab)

Reading a scrolling console is hard. The callback below records the key metrics
during training and plots them at the end with `matplotlib` &mdash; this runs
identically **locally and on Colab**, no external tools needed.

We train a fresh model so the callback captures the whole run from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3.common.callbacks import BaseCallback


class MetricLogger(BaseCallback):
    """Collect selected metrics from SB3's logger after each rollout."""

    def __init__(self):
        super().__init__()
        self.records = {"timesteps": [], "ep_rew_mean": [], "approx_kl": [],
                        "explained_variance": [], "entropy_loss": []}

    def _on_step(self) -> bool:
        return True  # required; we log per rollout, not per step

    def _on_rollout_end(self) -> None:
        logs = self.logger.name_to_value
        # ep_rew_mean is read from the episode-info buffer directly, because the
        # logger key is only populated once completed episodes are available.
        ep_buffer = self.model.ep_info_buffer
        ep_rew = np.mean([e["r"] for e in ep_buffer]) if len(ep_buffer) > 0 else np.nan
        self.records["timesteps"].append(self.num_timesteps)
        self.records["ep_rew_mean"].append(ep_rew)
        self.records["approx_kl"].append(logs.get("train/approx_kl", np.nan))
        self.records["explained_variance"].append(logs.get("train/explained_variance", np.nan))
        self.records["entropy_loss"].append(logs.get("train/entropy_loss", np.nan))


env = gym.make("CartPole-v1")
logger_cb = MetricLogger()
model = PPO("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=100_000, callback=logger_cb)

r = logger_cb.records
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
panels = [
    ("ep_rew_mean", "Mean episodic reward", 500, "target = 500"),
    ("approx_kl", "approx_kl", 0.02, "healthy < ~0.02"),
    ("explained_variance", "explained_variance", 1.0, "target -> 1.0"),
    ("entropy_loss", "entropy_loss", 0.0, "-> 0 as policy sharpens"),
]
for ax, (key, title, ref, note) in zip(axes.ravel(), panels):
    ax.plot(r["timesteps"], r[key])
    ax.axhline(ref, color="grey", ls="--", alpha=0.7, label=note)
    ax.set_title(title)
    ax.set_xlabel("timesteps")
    ax.legend(fontsize=8)
    ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### Optional: TensorBoard

Because the training cell passes `tensorboard_log="./ppo_cartpole_tb/"`, you can
also inspect the curves in TensorBoard.

- **Locally**: run `tensorboard --logdir ./ppo_cartpole_tb/` in a terminal and
  open the printed URL.
- **On Colab**: run the two lines below in a cell.

```python
%load_ext tensorboard
%tensorboard --logdir ./ppo_cartpole_tb/
```

The inline matplotlib plots above already cover the essentials, so TensorBoard
is optional here.

### Exercise 3: Watch a trained episode

Test the trained agent on one CartPole episode.

#### Run this if you use a local Python setup:

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

env = gym.make("CartPole-v1", render_mode="human")

# Load the model and run one episode
model = PPO.load("ppo_cartpole")

obs, _ = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    env.render()

env.close()


#### Run this if you use Colab (renders to a video):

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

from IPython.display import HTML
from base64 import b64encode

import imageio

# instantiation of the environment
env = gym.make("CartPole-v1", render_mode="rgb_array")

# resetting the environment for first start
obs, _ = env.reset()

# initialize a list of frames for video creation
frames = []

done = False
while not done:
    # capture the frame and append it to frames list
    frame = env.render()
    frames.append(frame)

    action, _ = model.predict(obs)
    # do one step in the environment
    obs, reward, terminated, truncated, info = env.step(action)

    # flag whether the episode is finished
    done = terminated or truncated

    # final rendering for last image of episode
    if done:
      frame = env.render()
      frames.append(frame)

env.close()

# save video as
video_path = "./CartPole_vid_own_policy.mp4"
imageio.mimsave(video_path, frames, fps=5)


In [ ]:
# this is for displaying the video after saving
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")


### Exercise 4: Extensions &mdash; network architecture and policy types

#### a) Tuning the MLP architecture

The default `MlpPolicy` uses two hidden layers of 64 units each, *separate* for
the policy (`pi`) and value (`vf`) networks. You can change this via
`policy_kwargs`:

```python
policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
model = PPO("MlpPolicy", env, policy_kwargs=policy_kwargs, verbose=1)
```

For CartPole the default is already plenty; larger networks mostly help on
higher-dimensional observations.

#### b) MLP vs. LSTM policy &mdash; when do you need recurrence?

An `MlpPolicy` maps the **current** observation to an action. This is enough
when the observation is **Markovian** (fully describes the state). CartPole is
fully observable &mdash; position, velocity, angle and angular velocity are all
in the observation &mdash; so an MLP is the right choice and an LSTM buys you
nothing here.

You need a **recurrent (LSTM) policy** when the environment is *partially
observable* (a POMDP): the agent must remember past observations to infer the
hidden state. Examples: velocity not included in the observation, noisy or
occluded sensors, or tasks requiring memory over time.

**Important:** plain `stable-baselines3` does **not** ship an LSTM policy. It
lives in the companion package **`sb3-contrib`** as `RecurrentPPO` with the
`"MlpLstmPolicy"`. The cell below shows how it would look &mdash; it is **not
run here** because `sb3-contrib` is an extra dependency and CartPole does not
need it. Install it with `pip install sb3-contrib` to try it out.

In [ ]:
# OPTIONAL -- requires: pip install sb3-contrib
# Recurrent PPO is only worthwhile for partially observable environments.
from sb3_contrib import RecurrentPPO

env = gym.make("CartPole-v1")
model = RecurrentPPO("MlpLstmPolicy", env, verbose=1)
model.learn(total_timesteps=100_000)

# Note: prediction must carry the LSTM hidden state between steps:
obs, _ = env.reset()
lstm_states = None
episode_start = True
done = False
while not done:
    action, lstm_states = model.predict(obs, state=lstm_states,
                                        episode_start=episode_start)
    obs, reward, terminated, truncated, _ = env.step(action)
    episode_start = terminated or truncated
    done = terminated or truncated
